# Workflow or agent?

**Thursday, last block.**

Everything so far had you in control: you wrote the prompt, you decided what
went into the context, you chose when to call the model. That is a
**workflow** - you write the control flow, the model fills in the steps.

An **agent** moves one thing: the model decides the control flow. It chooses
what to do next, and it keeps going until it decides it is done.

That single change buys flexibility and costs almost everything else -
predictability, cost, testability. This notebook builds both on the same task
so the trade is visible rather than asserted.

The rule worth taking home: **use a workflow unless an agent is genuinely
necessary.** Most of what people call agents today is a workflow with one LLM
call in the middle, and that is good news, not a criticism.

In [ ]:
# --- SETUP: run this first ---  [lares-setup-v1]
# works in Colab and locally; safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/hrvojenovak/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

## Part 0: Setup

`task="agent"` picks the chain whose bottleneck is requests per day and tool
calling, not context size. An agent loop makes 5-15 calls per run, so the
daily quota is what runs out first.

In [ ]:
import json, time, io, contextlib, traceback, re

API_KEY = None
if "google.colab" in sys.modules:
    from google.colab import userdata
    try:
        API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception as e:
        print("secret not available:", type(e).__name__)
else:
    API_KEY = os.environ.get("GOOGLE_API_KEY")

TASK = "agent"

from lares_llm import (set_key, ask, call, usage, which, supports,
                       CHAINS, LIMITS, STATS, LAST)
set_key(API_KEY)

print("key:", "OK" if API_KEY else "MISSING - see notebook 9, Part 0")
print(f"chain for '{TASK}': {' > '.join(CHAINS[TASK])}")
print(f"using: {which(TASK)}  {LIMITS.get(which(TASK), {})}")

## Part 1: The task

Same job for both approaches, on the dataset the week has been using:

> Load the house price data, find the three features most correlated with the
> sale price, check them for missing values, and report the result.

Chosen deliberately: it has **known correct answers**, so we can score both
approaches rather than admire them.

In [ ]:
DATA = "../data/housing_prices/housing.csv"
print("data file:", DATA, "|", "found" if os.path.exists(DATA) else "NOT FOUND")
if not os.path.exists(DATA):
    print("\navailable csv files:")
    import glob
    for f in sorted(glob.glob("../data/**/*.csv", recursive=True))[:10]:
        print("  ", f)

TASK_TEXT = ("Load the house price data, find the three numeric features most "
             "correlated with SalePrice, and check those three for missing "
             "values. Report the result.")

## Part 2: The workflow

Three fixed steps. **We** wrote the order; the model only fills in one of
them.

Notice what this gives us: the number of LLM calls is known in advance, the
code that touches the data is ours, and the whole thing can be unit tested.
Notice also what it costs: if the question changes, we rewrite the pipeline.

In [ ]:
import pandas as pd, numpy as np

def workflow(path: str) -> dict:
    """fixed pipeline: our code does the work, the model only writes the summary"""
    t0 = time.time()
    calls_before = STATS["requests"]

    # step 1 - our code, deterministic
    df = pd.read_csv(path)
    num = df.select_dtypes(include=np.number).drop(columns=["SalePrice"], errors="ignore")
    corr = num.corrwith(df["SalePrice"]).abs().sort_values(ascending=False)
    top3 = corr.head(3)

    # step 2 - our code, deterministic
    missing = df[list(top3.index)].isna().sum().to_dict()

    # step 3 - the ONE place the model is used: turning numbers into a sentence
    facts = {"top3": {k: round(float(v), 3) for k, v in top3.items()},
             "missing": missing, "rows": len(df)}
    summary = ask(f"Write two sentences summarising these findings for an "
                  f"engineer. Use the numbers as given.\n\n{json.dumps(facts)}",
                  task=TASK, max_tokens=300)

    return {"facts": facts, "summary": summary,
            "llm_calls": STATS["requests"] - calls_before,
            "seconds": time.time() - t0}


wf = workflow(DATA)
print(json.dumps(wf["facts"], indent=1))
print("\nsummary:", wf["summary"])
print(f"\nLLM calls: {wf['llm_calls']}  |  {wf['seconds']:.1f}s")

## Part 3: The agent

Now the same job, but the model decides. We give it one tool and a loop, and
step out of the way.

This is the whole thing, in about forty lines. Three pieces:

- **a tool** - a name, a description, and a JSON schema for its arguments
- **a sandbox** - our code that actually executes what the model asks for
- **a loop** - call, execute, feed the result back, call again

Look at the schema. It is the structured output from notebook 9, Part 4, under
a different name. A tool call *is* structured output.

In [ ]:
TOOL = {
    "name": "run_python",
    "description": ("Run Python code. State persists between calls, so variables "
                    "stay. Use print() to see anything - only stdout is returned. "
                    "pandas is available as pd, numpy as np. No internet access."),
    "parameters": {
        "type": "object",
        "properties": {"code": {"type": "string", "description": "Python code"}},
        "required": ["code"],
    },
}


class Sandbox:
    """persistent namespace, with output truncation"""

    def __init__(self, max_chars: int = 3000):
        self.ns = {"pd": pd, "np": np}
        self.max_chars = max_chars
        self.calls = 0

    def __call__(self, code: str) -> str:
        self.calls += 1
        buf = io.StringIO()
        try:
            with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                exec(code, self.ns)
        except Exception:
            # the traceback goes BACK to the model - that is how it self-corrects
            buf.write("\n--- TRACEBACK ---\n" + traceback.format_exc(limit=3))

        out = buf.getvalue() or "(no output - use print() to see something)"
        if len(out) > self.max_chars:
            # one print(df) can otherwise blow the whole context window
            head, tail = out[:self.max_chars // 2], out[-self.max_chars // 2:]
            out = f"{head}\n... [{len(out) - self.max_chars} chars cut] ...\n{tail}"
        return out

In [ ]:
AGENT_SYSTEM = ("You are a data analyst. You have one tool, run_python. "
                "Work step by step, checking each result before the next step. "
                "When you are finished, reply with plain text and NO tool call.")


def agent(task_text: str, max_iter: int = 12, verbose: bool = True) -> dict:
    """the loop: call, execute, feed back, repeat"""
    sb = Sandbox()
    contents = [{"role": "user", "parts": [{"text": task_text}]}]
    t0, calls_before = time.time(), STATS["requests"]
    diag = {"turns": 0, "tool_calls": 0, "malformed": 0, "stopped": "max_iter",
            "answer": "", "peak_input": 0}

    for turn in range(1, max_iter + 1):
        diag["turns"] = turn
        res = call(contents, tools=[TOOL], task=TASK, system=AGENT_SYSTEM,
                   max_tokens=2048, verbose=verbose)
        diag["peak_input"] = max(diag["peak_input"], LAST["input"])

        if not res["ok"]:
            diag["stopped"] = "error"
            diag["answer"] = res["text"]
            break

        # no tool call -> the model considers itself done
        if not res["tool_calls"]:
            diag["stopped"] = "done"
            diag["answer"] = res["text"]
            if verbose:
                print(f"  [{turn}] finished")
            break

        contents.append({"role": "model", "parts": res["raw_parts"]})

        for tc in res["tool_calls"]:
            diag["tool_calls"] += 1
            code = (tc.get("args") or {}).get("code")
            if not isinstance(code, str) or not code.strip():
                diag["malformed"] += 1
                out = "ERROR: missing string argument 'code'."
                if verbose:
                    print(f"  [{turn}] MALFORMED tool call")
            else:
                out = sb(code)
                if verbose:
                    print(f"  [{turn}] run_python ({len(code)} chars) -> "
                          f"{out.strip()[:60]!r}")
            contents.append({"role": "user", "parts": [{"functionResponse": {
                "name": tc["name"], "response": {"output": out}}}]})

    diag["llm_calls"] = STATS["requests"] - calls_before
    diag["seconds"] = time.time() - t0
    diag["sandbox_calls"] = sb.calls
    return diag


ag = agent(f"{TASK_TEXT} The file is at {DATA!r}.")
print("\nanswer:", ag["answer"][:400])

## Part 4: The comparison

Same task, both ways. Run the cell and look at the columns, not at the prose.

In [ ]:
print(f"{'':22s} {'workflow':>12s} {'agent':>12s}")
print("-" * 50)
rows = [("LLM calls", wf["llm_calls"], ag["llm_calls"]),
        ("seconds", f"{wf['seconds']:.1f}", f"{ag['seconds']:.1f}"),
        ("code we wrote", "all of it", "the loop"),
        ("code the model wrote", "none", f"{ag['sandbox_calls']} blocks"),
        ("steps known upfront", "yes", "no"),
        ("unit testable", "yes", "not really"),
        ("survives a new question", "no", "yes")]
for label, a, b in rows:
    print(f"{label:22s} {str(a):>12s} {str(b):>12s}")

print(f"\nagent finished on its own: {ag['stopped'] == 'done'}")
print(f"malformed tool calls: {ag['malformed']}")
usage("last call")

The agent cost several times more calls and time to reach the same answer,
and the workflow cannot be wrong about the correlations because *our* code
computed them.

On this task the workflow wins on every axis that matters. That is the
expected result, and it is the point.

## Part 5: The task where the workflow breaks

Change the question:

> Find something unusual in this dataset.

Now write the pipeline. You cannot - not because it is hard, but because you
do not know which steps to write. "Unusual" is not a computation until you
have looked.

This is the boundary. An agent is worth its cost when **you cannot enumerate
the steps in advance**, and not before.

In [ ]:
OPEN_TASK = (f"Look at the dataset at {DATA!r} and find something unusual or "
             "surprising about it. Investigate whatever you think is worth "
             "checking, then tell me what you found and why it matters.")

ag2 = agent(OPEN_TASK, max_iter=12)
print("\n=== what it found ===")
print(ag2["answer"][:900])
print(f"\nturns: {ag2['turns']}  |  tool calls: {ag2['tool_calls']}  |  "
      f"{ag2['seconds']:.1f}s  |  peak input: {ag2['peak_input']:,} tokens")

Two things to judge, and they are different questions:

1. **Is the finding real?** Check it yourself - the agent is not a witness.
2. **Would you have found it?** If yes, the agent saved you time. If no, it did
   something a workflow could not.

Also worth watching: how many of its steps were *wasted*. An agent that took
nine turns to do what three would have done is telling you the task was
actually enumerable.

## Part 6: Why it fails, and how the failures compound

An agent is a chain of steps that each work most of the time. That is worse
than it sounds.

If each step is 90 % reliable, five steps end to end are `0.9^5` = **59 %**.
Nothing was buggy; the arithmetic just is what it is.

In [ ]:
print(f"{'steps':>6s}  " + "  ".join(f"{p:.0%}" for p in (0.99, 0.95, 0.90, 0.80)))
print("-" * 40)
for n in (1, 3, 5, 10, 20):
    print(f"{n:>6d}  " + "  ".join(f"{p**n:>4.0%}" for p in (0.99, 0.95, 0.90, 0.80)))

print("\n-> this is the argument for fewer steps, not for a better model")
print("-> and the argument for a human checkpoint before anything irreversible")

### The guardrails already in this notebook

Four of them, and each exists because of a specific failure:

**`max_iter`** - an agent that never emits a final answer will loop until the
quota is gone. Try setting it to 2 and watch what happens.

**Output truncation in the sandbox** - one `print(df)` on a wide table sends
tens of thousands of tokens back into the context. That is the fastest way to
hit a token-per-minute limit, and it looks like the model being slow.

**Tracebacks go back to the model** - this is a guardrail *for* the agent, not
against it. Handing back the error is what lets it self-correct, and it is why
the loop needs no retry logic of its own.

**One tool, and it has no side effects** - `run_python` reads files and prints.
It sends no email, writes no database. In a question-answering system the worst
case is a wrong answer; **in an agent with tools the worst case is an
action.**

In [ ]:
# deliberately break it: no room to finish
short = agent(f"{TASK_TEXT} The file is at {DATA!r}.", max_iter=2, verbose=False)
print(f"max_iter=2  -> stopped: {short['stopped']}, turns: {short['turns']}")
print(f"             answer: {short['answer'][:80]!r}")

# and what an untruncated tool result would have cost
sb_test = Sandbox(max_chars=3000)
big = sb_test(f"import pandas as pd; print(pd.read_csv({DATA!r}).head(20).to_string())")
raw_len = len(pd.read_csv(DATA).head(20).to_string())
print(f"\nuntruncated tool output: {raw_len:,} chars (~{raw_len//3.7:,.0f} tokens)")
print(f"after truncation:        {len(big):,} chars")

## Part 7: Optional - a second tool

At 13:00 we wrapped YOLO into `detect(path) -> list[dict]` and said the agent
would get it. Here it is.

Nothing about the loop changes. We add a second entry to the tool list and one
branch in the dispatch. That is the point worth seeing: **to the agent, a tool
is anything with a defined interface** - text, an image detector, a database
query, a REST call.

Skip this cell if you are short on time; it downloads YOLO weights again.

In [ ]:
RUN_YOLO_TOOL = False        # set to True to try it

if RUN_YOLO_TOOL:
    %pip install -q ultralytics
    os.environ["YOLO_VERBOSE"] = "False"
    from ultralytics import YOLO
    yolo = YOLO("yolo11n.pt")

    DETECT_TOOL = {
        "name": "detect",
        "description": "Detect objects in an image. Returns labels with confidence.",
        "parameters": {"type": "object",
                       "properties": {"path": {"type": "string"}},
                       "required": ["path"]},
    }

    def detect(path: str) -> list[dict]:
        if not os.path.exists(path):
            return [{"error": f"no such file: {path}"}]
        r = yolo(path, verbose=False)[0]
        return [{"label": yolo.names[int(b.cls)],
                 "confidence": round(float(b.conf), 3)} for b in r.boxes]

    TOOLS = {"run_python": lambda a: Sandbox()(a.get("code", "")),
             "detect": lambda a: json.dumps(detect(a.get("path", "")))}
    print("two tools registered:", list(TOOLS))
    print("the loop is unchanged - only the dispatch has one more branch")
else:
    print("skipped (set RUN_YOLO_TOOL = True to run)")

## Exercise: log three ways it broke

This is the part that makes the afternoon education rather than a demo.

Run the agent a few more times - change the task, lower `max_iter`, ask for
something the data cannot answer, ask in Croatian. **Write down three distinct
ways it failed**, and for each one:

- what you saw
- what caused it
- what would have caught it in production

Failure modes worth expecting: it never emits a final answer; it invents a
column name; it "verifies" something without running any code; it reports a
number that its own tool output contradicts; it succeeds but takes nine turns
to do three turns of work.

If you leave today thinking agents work, the afternoon has failed. If you
leave with three concrete failures and a guardrail for each, it has not.

In [ ]:
# space for your notes
failures = [
    # {"saw": "...", "cause": "...", "guardrail": "..."},
]

for i, f in enumerate(failures, 1):
    print(f"{i}. {f['saw']}\n   cause: {f['cause']}\n   guardrail: {f['guardrail']}")
print(f"\n{len(failures)} logged")

## Takeaways

**A workflow is you writing the control flow; an agent is the model writing
it.** Everything else follows from that one difference.

**Use a workflow unless you cannot enumerate the steps.** Predictable, cheap,
testable. Most production "agents" are workflows, and that is fine.

**An agent loop is about forty lines.** Call, execute, feed the result back,
repeat. No framework required to understand it - and frameworks are worth
adding only once you know what they are hiding.

**A tool call is structured output.** Same mechanism as notebook 9, Part 4.

**Reliability compounds downwards.** Five steps at 90 % is 59 %. Fewer steps
beats a better model.

**The worst case scales with the tools.** Answering questions: a wrong answer.
Holding tools with side effects: a wrong action. Give the least privilege that
does the job, and keep a human in front of anything irreversible.

---

Tomorrow morning: what has to be true for any of this to survive contact with
a real organisation.